# StayNest - Session 7 Assignment (Delta Lake & Lakehouse)
Work through the 8 tasks in order. Read the Assignment Questions PDF for the full
detail and acceptance criteria. Fill each `# TODO` cell, run it, and keep the output
visible. Runs on Databricks Free Edition (serverless).

## Section 0 - Setup (already done for you)
Upload `bookings.csv`, `hotels.csv`, `bookings_updates.csv` to a Volume, set `BASE`,
`CATALOG`, `SCHEMA`, and run this cell. Expect 12000 / 200 / 200.

In [0]:
BASE    = "/Volumes/workspace/default/staynest_s07_assignment"
CATALOG = "workspace"
SCHEMA  = "default"
FQN = lambda name: f"{CATALOG}.{SCHEMA}.{name}"

read_csv = lambda name: (spark.read
    .option("header", True).option("inferSchema", True)
    .csv(f"{BASE}/{name}.csv"))

bookings_df = read_csv("bookings")
hotels_df   = read_csv("hotels")
updates_df  = read_csv("bookings_updates")

print(f"bookings: {bookings_df.count()}, hotels: {hotels_df.count()}, "
      f"updates: {updates_df.count()}")

bookings: 12000, hotels: 200, updates: 200


## Task 1 - Read the plan and force a broadcast join
Join bookings to hotels and call `.explain()` to see the plan. Then force a
broadcast join with `broadcast(hotels_df)` and `.explain()` again. In a comment,
say which join each plan used and why broadcast avoids a shuffle.
(Tip: hotels also has a `city` column, so `hotels_df.drop("city")` before joining.)

In [0]:
from pyspark.sql.functions import broadcast

joined_df = bookings_df.join(hotels_df, on="hotel_id", how="inner")
joined_df.explain()

broadcast_joined_df = bookings_df.join(broadcast(hotels_df), on="hotel_id", how="inner")
broadcast_joined_df.explain()

# Regular join uses SortMergeJoin.
# Forced broadcast join uses BroadcastHashJoin.

# In databricks, config 'spark.sql.autoBroadcastJoinThreshold' is set. It cannot be changed because it is limitation of serverless cluster. The output therefore shows PhotonBroadcastHashJoin in both plans. 
# The second text output shows different plans because I disabled this configuration in local spark in my machine and than ran both joins.

# Broadcast avoids the shuffle because the broadcasted dataframe is sent to all active executors and then the join is performed locally on each executor. 


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [hotel_id#21769, booking_id#21767, customer_id#21768, booking_date#21770, city#21771, nights#21772, amount#21773, status#21774, hotel_name#21799, city#21800, category#21801, star_rating#21802]
         +- PhotonBroadcastHashJoin [hotel_id#21769], [hotel_id#21798], Inner, BuildRight, false, true
            :- PhotonFilter isnotnull(hotel_id#21769)
            :  +- PhotonRowToColumnar
            :     +- FileScan csv [booking_id#21767,customer_id#21768,hotel_id#21769,booking_date#21770,city#21771,nights#21772,amount#21773,status#21774] Batched: false, DataFilters: [isnotnull(hotel_id#21769)], Format: CSV, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/workspace/default/staynest_s07_assignment/bookings.csv], PartitionFilters: [], PushedFilters: [IsNotNull(hotel_id)], ReadSchema: struct<booking_id:int,customer_id:int,hotel_id:int,booking

In [0]:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [hotel_id#115, booking_id#113, customer_id#114, booking_date#116, city#117, nights#118, amount#119, status#120, hotel_name#139, city#140, category#141, star_rating#142]
   +- SortMergeJoin [hotel_id#115], [hotel_id#138], Inner
      :- Sort [hotel_id#115 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(hotel_id#115, 200), ENSURE_REQUIREMENTS, [plan_id=348]
      :     +- Filter isnotnull(hotel_id#115)
      :        +- FileScan csv [booking_id#113,customer_id#114,hotel_id#115,booking_date#116,city#117,nights#118,amount#119,status#120] Batched: false, DataFilters: [isnotnull(hotel_id#115)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/shalinigoutam/VS Code Projects/PYSPARK PRACTICE/PySpark_Tu..., PartitionFilters: [], PushedFilters: [IsNotNull(hotel_id)], ReadSchema: struct<booking_id:int,customer_id:int,hotel_id:int,booking_date:date,city:string,nights:int,amoun...
      +- Sort [hotel_id#138 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(hotel_id#138, 200), ENSURE_REQUIREMENTS, [plan_id=349]
            +- Filter isnotnull(hotel_id#138)
               +- FileScan csv [hotel_id#138,hotel_name#139,city#140,category#141,star_rating#142] Batched: false, DataFilters: [isnotnull(hotel_id#138)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/shalinigoutam/VS Code Projects/PYSPARK PRACTICE/PySpark_Tu..., PartitionFilters: [], PushedFilters: [IsNotNull(hotel_id)], ReadSchema: struct<hotel_id:int,hotel_name:string,city:string,category:string,star_rating:double>


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [hotel_id#115, booking_id#113, customer_id#114, booking_date#116, city#117, nights#118, amount#119, status#120, hotel_name#139, city#140, category#141, star_rating#142]
   +- BroadcastHashJoin [hotel_id#115], [hotel_id#138], Inner, BuildRight, false
      :- Filter isnotnull(hotel_id#115)
      :  +- FileScan csv [booking_id#113,customer_id#114,hotel_id#115,booking_date#116,city#117,nights#118,amount#119,status#120] Batched: false, DataFilters: [isnotnull(hotel_id#115)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/shalinigoutam/VS Code Projects/PYSPARK PRACTICE/PySpark_Tu..., PartitionFilters: [], PushedFilters: [IsNotNull(hotel_id)], ReadSchema: struct<booking_id:int,customer_id:int,hotel_id:int,booking_date:date,city:string,nights:int,amoun...
      +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=380]
         +- Filter isnotnull(hotel_id#138)
            +- FileScan csv [hotel_id#138,hotel_name#139,city#140,category#141,star_rating#142] Batched: false, DataFilters: [isnotnull(hotel_id#138)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/shalinigoutam/VS Code Projects/PYSPARK PRACTICE/PySpark_Tu..., PartitionFilters: [], PushedFilters: [IsNotNull(hotel_id)], ReadSchema: struct<hotel_id:int,hotel_name:string,city:string,category:string,star_rating:double>


## Task 2 - Create a Delta table, then read its history
Write `bookings_df` as a managed Delta table with `saveAsTable`. Then create some
history: run an `UPDATE` (set pending to completed) and a `DELETE` (remove
cancelled). Show `DESCRIBE HISTORY` and point out the versioned commits.

In [0]:
(
    bookings_df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("bookings_delta")
)

spark.sql(" UPDATE bookings_delta SET status = 'completed' WHERE status = 'pending' ")

spark.sql(" DELETE FROM bookings_delta WHERE status = 'cancelled' ")

spark.sql(" DESCRIBE HISTORY bookings_delta ").select("version", "timestamp", "operation").show()




+-------+-------------------+--------------------+
|version|          timestamp|           operation|
+-------+-------------------+--------------------+
|      2|2026-07-19 14:05:11|              DELETE|
|      1|2026-07-19 14:05:08|              UPDATE|
|      0|2026-07-19 14:05:04|CREATE OR REPLACE...|
+-------+-------------------+--------------------+



## Task 3 - Time travel and RESTORE
Read the table as it was at **version 0** (before your UPDATE and DELETE) and show
its count. Then `RESTORE` the table to version 0 and confirm the count is back.
Show that RESTORE appears as a new commit in the history.

In [0]:
(
    spark.read
        .format("delta")
        .option("versionAsOf", 0)
        .table("bookings_delta")
        .count()
    
)
spark.sql("""SELECT  
                (SELECT COUNT(*) FROM bookings_delta VERSION AS OF 0) AS v0, 
                (SELECT COUNT(*) FROM bookings_delta ) AS current_rows
          """).show()

spark.sql("RESTORE TABLE bookings_delta TO VERSION AS OF 0 ")

spark.sql("""SELECT  
                (SELECT COUNT(*) FROM bookings_delta VERSION AS OF 0) AS v0, 
                (SELECT COUNT(*) FROM bookings_delta ) AS restored_rows
          """).show()

spark.sql(" DESCRIBE HISTORY bookings_delta ").select("version", "timestamp", "operation").show()

# Here we received restore operation after changing the version pointer. Optimize entry is auto optimize performed by spark.

+-----+------------+
|   v0|current_rows|
+-----+------------+
|12000|       10563|
+-----+------------+

+-----+-------------+
|   v0|restored_rows|
+-----+-------------+
|12000|        12000|
+-----+-------------+

+-------+-------------------+--------------------+
|version|          timestamp|           operation|
+-------+-------------------+--------------------+
|      4|2026-07-19 14:05:36|             RESTORE|
|      3|2026-07-19 14:05:13|            OPTIMIZE|
|      2|2026-07-19 14:05:11|              DELETE|
|      1|2026-07-19 14:05:08|              UPDATE|
|      0|2026-07-19 14:05:04|CREATE OR REPLACE...|
+-------+-------------------+--------------------+



## Task 4 - OPTIMIZE and ZORDER
Run `OPTIMIZE` on your Delta table to compact files. Then run
`OPTIMIZE ... ZORDER BY (city)`. In a comment, say what OPTIMIZE does and why
`city` is a good ZORDER column but `status` would not be.

In [0]:

spark.sql(" OPTIMIZE bookings_delta ")

spark.sql(" OPTIMIZE bookings_delta ZORDER BY (city) ")

spark.sql(" DESCRIBE HISTORY bookings_delta ").select("version", "timestamp", "operation").show()


# OPTIMIZE compacts many small files into few big ones. 
# Here we do not see additional optimize operation because spark had done auto optimize and now there is nothing to optimize.
# city is a good ZORDER column but not status because city has high cardinality (many distinct values) whereas status doesn't.


+-------+-------------------+--------------------+
|version|          timestamp|           operation|
+-------+-------------------+--------------------+
|      4|2026-07-19 14:05:36|             RESTORE|
|      3|2026-07-19 14:05:13|            OPTIMIZE|
|      2|2026-07-19 14:05:11|              DELETE|
|      1|2026-07-19 14:05:08|              UPDATE|
|      0|2026-07-19 14:05:04|CREATE OR REPLACE...|
+-------+-------------------+--------------------+



## Task 5 - Bronze: land the raw data
Write the raw bookings to a `bronze_bookings` Delta table, keeping every row and
adding an `ingested_at` timestamp column.

In [0]:
from pyspark.sql.functions import current_timestamp

bronze_bookings = (
                    spark.read
                        .option("header", True)
                        .option("inferSchema", True)
                        .csv(f"{BASE}/bookings.csv")
                        .withColumn("ingested_at", current_timestamp())
                )


(
    bronze_bookings.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("bronze_bookings")
)

spark.sql("SELECT * from bronze_bookings").show(10, truncate=False)

+----------+-----------+--------+------------+---------+------+--------+---------+--------------------------+
|booking_id|customer_id|hotel_id|booking_date|city     |nights|amount  |status   |ingested_at               |
+----------+-----------+--------+------------+---------+------+--------+---------+--------------------------+
|9000000   |701600     |3095    |2025-11-27  |Jaipur   |4     |6087.65 |completed|2026-07-19 14:08:22.153417|
|9000001   |700065     |3057    |2025-11-06  |Delhi    |1     |8211.19 |cancelled|2026-07-19 14:08:22.153417|
|9000002   |701392     |3187    |2025-08-21  |Jaipur   |2     |7176.52 |cancelled|2026-07-19 14:08:22.153417|
|9000003   |700867     |3112    |2025-03-22  |Bengaluru|5     |7880.62 |completed|2026-07-19 14:08:22.153417|
|9000004   |701521     |3043    |2025-04-19  |Mumbai   |5     |21021.51|pending  |2026-07-19 14:08:22.153417|
|9000005   |701998     |3018    |2025-10-14  |Mumbai   |2     |18124.49|cancelled|2026-07-19 14:08:22.153417|
|9000006  

## Task 6 - Silver: clean and conform
Build `silver_bookings` from bronze: keep only completed bookings and join the
hotel dimension to add `category`, `star_rating`, and the hotel name. Drop the
duplicate `city` from the hotel side so the join has a single `city`.

In [0]:
from pyspark.sql.functions import col

hotels_df = hotels_df.drop("city")
silver_bookings = (
                    spark.table("bronze_bookings")
                        .filter(col("status") == "completed")
                        .join(hotels_df, on="hotel_id", how="inner")
                )

(
    silver_bookings.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("silver_bookings")
)

spark.sql("SELECT * from silver_bookings").show(10, truncate=False)


+--------+----------+-----------+------------+---------+------+--------+---------+--------------------------+----------------+--------+-----------+
|hotel_id|booking_id|customer_id|booking_date|city     |nights|amount  |status   |ingested_at               |hotel_name      |category|star_rating|
+--------+----------+-----------+------------+---------+------+--------+---------+--------------------------+----------------+--------+-----------+
|3095    |9000000   |701600     |2025-11-27  |Jaipur   |4     |6087.65 |completed|2026-07-19 14:08:22.153417|Orchid Suites   |Budget  |3.8        |
|3112    |9000003   |700867     |2025-03-22  |Bengaluru|5     |7880.62 |completed|2026-07-19 14:08:22.153417|Grand Stay      |Budget  |3.6        |
|3012    |9000006   |701336     |2025-11-25  |Delhi    |7     |70999.15|completed|2026-07-19 14:08:22.153417|Orchid Residency|Luxury  |4.1        |
|3127    |9000007   |700868     |2025-04-10  |Mumbai   |2     |15693.64|completed|2026-07-19 14:08:22.153417|Sum

## Task 7 - Gold: business-ready aggregate
From silver, build a `gold_city_revenue` Delta table: bookings and total revenue
per city, ordered by revenue.

In [0]:
from pyspark.sql.functions import col, sum, round


gold_city_revenue_df = ( spark.table("silver_bookings")
                            .filter(col("status") == "completed")
                            .groupBy(col("city"))
                            .agg(round(sum(col("amount")),2).alias('total_revenue_per_city'))
                            .orderBy(col('total_revenue_per_city').desc())

                        )

(
    gold_city_revenue_df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("gold_city_revenue")
)

spark.sql("SELECT * from gold_city_revenue").display(10, truncate=False)



city,total_revenue_per_city
Goa,4.459670179E7
Mumbai,3.624122112E7
Delhi,2.631428154E7
Jaipur,2.443685313E7
Bengaluru,2.267013697E7
Udaipur,1.209442742E7
Rishikesh,8606121.58
Manali,6235480.68
Munnar,3979216.11
Anantapur,2257080.65


## Task 8 - Incremental load with MERGE
You have today's batch in `updates_df` (150 changed bookings + 50 new ones).
`MERGE` it into your Delta table: update matched booking_ids, insert new ones, in
one command. Report the row count before and after (it should grow by the 50 new).

In [0]:
from pyspark.sql.functions import current_timestamp

before = spark.sql("SELECT * FROM bronze_bookings").count()

updates_df.createOrReplaceTempView("updates_df")

spark.sql("""
          MERGE INTO bronze_bookings AS tgt 
          USING updates_df AS src
          ON src.booking_id = tgt.booking_id

          WHEN MATCHED THEN 
          UPDATE SET 
            booking_id = src.booking_id,
            customer_id = src.customer_id,
            hotel_id = src.hotel_id,
            booking_date = src.booking_date,
            city = src.city,
            nights = src.nights,
            amount = src.amount,
            status = src.status


          WHEN NOT MATCHED THEN 
          INSERT 
            (   booking_id,
                customer_id,
                hotel_id,
                booking_date,
                city,
                nights,
                amount,
                status,
                ingested_at
            )
        VALUES (
                src.booking_id,
                src.customer_id,
                src.hotel_id,
                src.booking_date,
                src.city,
                src.nights,
                src.amount,
                src.status,
                current_timestamp()
                )
         """)

after = spark.sql("SELECT * FROM bronze_bookings").count()

print(f"Before re-run: {before:,}")
print(f"After re-run : {after:,}")


Before re-run: 12,000
After re-run : 12,050
